# TapeVision Play-Visualizer: Google Colab Showcase Batch-Rendering Workflow

This notebook orchestrates the batch generation of broadcast-grade visualization videos for the **50 finalized showcase clips** from the TapeVision American football dataset.

### Architecture Highlights
- **Sequential Batch Pipeline**: Renders one clip at a time into local Colab staging (`/content/play_visualizer_batch/staging`), validates the generated MP4 via `ffprobe`, and atomically commits the verified MP4 and report to persistent Google Drive.
- **Zero Duplicate Logic**: Leverages the official `play-visualizer-showcase-batch` CLI orchestration layer and core `render_annotation_video` engine.
- **Non-Destructive**: Never modifies source videos or annotation JSONs.
- **Resumable & Fault-Tolerant**: Independent clip execution with SHA-256 fingerprint matching so interruptions can be resumed instantly with `--resume`.

## Cell 1 — Configuration & Environment Settings
Define Google Drive paths, repository branch, scratch directory, and visualization settings.

In [ ]:
import os
from pathlib import Path

# 1. Repository Configuration
REPO_URL = "https://github.com/Alejandro-Fuste/Play-Visualizer.git"
BRANCH = "main"
LOCAL_REPO_DIR = Path("/content/Play-Visualizer")

# 2. Google Drive Data Roots
DRIVE_MOUNT_POINT = Path("/content/drive")
VIDEOS_ROOT = DRIVE_MOUNT_POINT / "MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/AmericanFootballDataset/videos"
ANNOTATIONS_ROOT = DRIVE_MOUNT_POINT / "MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Play-Visualizer/annotations"
OUTPUT_ROOT = DRIVE_MOUNT_POINT / "MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Play-Visualizer/outputs/showcase"

# Optional inventory CSV for file ID / provenance cross-checking
INVENTORY_CSV = DRIVE_MOUNT_POINT / "MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Play-Visualizer/1_Videos_Inventory.csv"

# 3. Scratch Staging Directory (fast local SSD in /content)
SCRATCH_ROOT = Path("/content/play_visualizer_batch")

# 4. Showcase Manifest (packaged in repository)
MANIFEST_PATH = LOCAL_REPO_DIR / "config/showcase_clips.json"

# 5. Representative Test Clips for initial QA
REPRESENTATIVE_TEST_CLIPS = [
    "JetSweep_1",
    "Post10to15_1",
    "Power_3",
    "PuntBlock_1",
    "KickoffReturnTouchdown_17",
    "Power_4",
]

# 6. Overlay and Visual Configuration
SHOW_TIMELINE = False
SHOW_OFFENSE = True
SHOW_DEFENSE = True
SHOW_ACTION_PANEL = True
INCLUDE_UNANNOTATED_TAIL = False

print("Configuration initialized successfully.")
print(f"Videos Root:      {VIDEOS_ROOT}")
print(f"Annotations Root: {ANNOTATIONS_ROOT}")
print(f"Output Root:      {OUTPUT_ROOT}")

## Cell 2 — Mount Google Drive
Mount your Google Drive to `/content/drive`. Google Colab will prompt you for authorization.

In [ ]:
from google.colab import drive

drive.mount(str(DRIVE_MOUNT_POINT))
print("Google Drive mounted at:", DRIVE_MOUNT_POINT)

## Cell 3 — Clone & Install Play-Visualizer
Clone the repository (or pull latest updates), install in editable mode with development dependencies, and verify FFmpeg and ffprobe.

In [ ]:
import subprocess
import sys

if not LOCAL_REPO_DIR.exists():
    print(f"Cloning Play-Visualizer ({BRANCH})...")
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(LOCAL_REPO_DIR)], check=True)
else:
    print("Repository already exists. Pulling latest updates...")
    subprocess.run(["git", "-C", str(LOCAL_REPO_DIR), "pull"], check=True)

# Install package in editable mode
print("Installing Play-Visualizer dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(LOCAL_REPO_DIR)], check=True)

# Verify FFmpeg & ffprobe
ffmpeg_res = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
ffprobe_res = subprocess.run(["ffprobe", "-version"], capture_output=True, text=True)

if ffmpeg_res.returncode == 0 and ffprobe_res.returncode == 0:
    print("FFmpeg and ffprobe are available and operational:")
    print("  -", ffmpeg_res.stdout.splitlines()[0])
    print("  -", ffprobe_res.stdout.splitlines()[0])
else:
    print("Installing ffmpeg via apt-get...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

## Cell 4 — Pre-flight Access Checks on Google Drive
Check that the configured videos directory and annotations directory exist on Google Drive, and verify baseline showcase clips.

In [ ]:
print("Checking Google Drive paths...")
print(f"Videos Root Exists:      {VIDEOS_ROOT.is_dir()} ({VIDEOS_ROOT})")
print(f"Annotations Root Exists: {ANNOTATIONS_ROOT.is_dir()} ({ANNOTATIONS_ROOT})")
print(f"Inventory CSV Exists:    {INVENTORY_CSV.is_file()} ({INVENTORY_CSV})")

baseline_checks = [
    ("JetSweep_1", "JetSweep"),
    ("Power_3", "Power"),
    ("PuntBlock_1", "PuntBlock"),
    ("KickoffReturnTouchdown_17", "KickoffReturnTouchdown"),
]

print("\nVerifying sample showcase files:")
all_found = True
for clip, play in baseline_checks:
    mp4_path = VIDEOS_ROOT / f"{clip}.mp4"
    json_path = ANNOTATIONS_ROOT / play / clip / "annotations.json"
    mp4_ok = mp4_path.is_file()
    json_ok = json_path.is_file()
    print(f"  [{clip:26s}] MP4: {'FOUND' if mp4_ok else 'MISSING'} | JSON: {'FOUND' if json_ok else 'MISSING'}")
    if not mp4_ok or not json_ok:
        all_found = False

if not all_found:
    print("\n[NOTICE] If annotation JSONs were uploaded as annotations.zip to Drive, extract them by running:")
    print('  !unzip -q /path/to/annotations.zip -d /path/to/target/folder -x "__MACOSX*"')

## Cell 5 — Full Showcase Preflight & Dry-Run (All 50 Clips)
Perform a full preflight audit across all 50 showcase clips: manifest validation, inventory cross-check, file existence, schema v1.1 compatibility, and stream probing.

**No MP4 files are rendered during dry-run.**

In [ ]:
cmd = [
    "play-visualizer-showcase-batch",
    "--videos-root", str(VIDEOS_ROOT),
    "--annotations-root", str(ANNOTATIONS_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--scratch-root", str(SCRATCH_ROOT),
    "--manifest", str(MANIFEST_PATH),
    "--dry-run",
]
if INVENTORY_CSV.exists():
    cmd.extend(["--inventory", str(INVENTORY_CSV)])

print("Running full showcase preflight dry-run...")
cmd_str = " ".join(cmd)
!{cmd_str}

## Cell 6 — Representative Showcase Subset Render (5–6 Clips)
Render a representative subset of showcase clips covering standard run plays, pass concepts, special teams, and repaired clips before initiating the full batch:
- `JetSweep_1` (Motion & Run)
- `Post10to15_1` (Pass Concept)
- `Power_3` & `Power_4` (Downhill Run & Repaired Clips)
- `PuntBlock_1` (Special Teams Defense)
- `KickoffReturnTouchdown_17` (Special Teams Scoring Return)

In [ ]:
subset_clips_arg = ",".join(REPRESENTATIVE_TEST_CLIPS)

cmd = [
    "play-visualizer-showcase-batch",
    "--videos-root", str(VIDEOS_ROOT),
    "--annotations-root", str(ANNOTATIONS_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--scratch-root", str(SCRATCH_ROOT),
    "--manifest", str(MANIFEST_PATH),
    "--clips", subset_clips_arg,
    "--resume",  # Safe: skips clips already rendered and verified
]
if SHOW_TIMELINE:
    cmd.append("--show-timeline")
else:
    cmd.append("--hide-timeline")

if INVENTORY_CSV.exists():
    cmd.extend(["--inventory", str(INVENTORY_CSV)])

print(f"Rendering representative subset ({len(REPRESENTATIVE_TEST_CLIPS)} clips)...")
cmd_str = " ".join(cmd)
!{cmd_str}

## Cell 7 — Full Showcase Batch Render (All 50 Clips)
Render all 50 showcase clips sequentially. Includes automatic staging in local scratch SSD, frame decodability checks, atomic transfer to Google Drive, and intermediate summary checkpoints.

> **Note:** `--resume` is enabled by default. If your Colab session disconnects or times out, re-running this cell will skip all previously completed and verified clips instantly.

In [ ]:
cmd = [
    "play-visualizer-showcase-batch",
    "--videos-root", str(VIDEOS_ROOT),
    "--annotations-root", str(ANNOTATIONS_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--scratch-root", str(SCRATCH_ROOT),
    "--manifest", str(MANIFEST_PATH),
    "--resume",  # Automatically resumes from where the run left off
]
if SHOW_TIMELINE:
    cmd.append("--show-timeline")
else:
    cmd.append("--hide-timeline")

if INVENTORY_CSV.exists():
    cmd.extend(["--inventory", str(INVENTORY_CSV)])

print("Beginning full 50-clip showcase batch rendering...")
cmd_str = " ".join(cmd)
!{cmd_str}

## Cell 8 — Inspect Batch Results & In-Notebook Video Player
Display the rendered batch summary report and play an annotated video directly inside the Colab notebook.

In [ ]:
import base64
from IPython.display import HTML, Markdown, display

# 1. Display Markdown Summary Report
summary_md_path = OUTPUT_ROOT / "showcase_batch_summary.md"
if summary_md_path.exists():
    display(Markdown(summary_md_path.read_text(encoding="utf-8")))
else:
    print("Batch summary report not found at:", summary_md_path)

# 2. Interactive Video Player Widget
preview_clip = "JetSweep_1"
preview_play = "JetSweep"
preview_video_path = OUTPUT_ROOT / preview_play / f"{preview_clip}.mp4"

if preview_video_path.exists():
    print(f"\nPreviewing {preview_clip} ({preview_video_path.stat().st_size / 1e6:.2f} MB):")
    video_data = preview_video_path.read_bytes()
    b64_video = base64.b64encode(video_data).decode("utf-8")
    html_player = f"""
    <video width="800" height="450" controls autoplay loop muted>
      <source src="data:video/mp4;base64,{b64_video}" type="video/mp4">
      Your browser does not support HTML5 video.
    </video>
    """
    display(HTML(html_player))
else:
    print(f"Preview video {preview_video_path} has not been rendered yet.")